# Cross-Domain Validation
**Test generalization to unseen generators/datasets**

## Held-out datasets (NEVER seen during training)
- **dall-e3**: DALL-E 3 generated images
- **celebdf-v2**: Celebrity face deepfakes
- **genimage-ai**: 21 generator types (BigGAN, StyleGAN, etc.)

In [ ]:
# Cell 0: Clone repo from GitHub
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/MIHMahmudEli/ai-image-detection-research.git"
CLONE_DIR = Path("/kaggle/working/ai-image-detection-research")

if not CLONE_DIR.exists():
    print(f"Cloning repo from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
    print("Clone complete")
else:
    print("Repo already cloned")

import os
os.chdir("/kaggle/working")
sys.path.insert(0, str(CLONE_DIR / "model"))
print(f"Project root: {CLONE_DIR}")

In [ ]:
# Cell 1: Setup
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.notebook import tqdm

sys.path.insert(0, str(Path.cwd().resolve() / 'ai-image-detection-research' / 'model'))
from src.kaggle_utils import KaggleEnv
env = KaggleEnv(project_root_search=True)
PROJECT_ROOT = env.project_root
os.chdir(env.working_dir)

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Cell 2: Load trained model
from src.model import build_mfft

VARIANT = 'base'  # Change to 'tiny' or 'large' as needed
model = build_mfft(VARIANT).to(device)

# Try to load from HF checkpoints
ckpt_path = PROJECT_ROOT / 'model' / 'checkpoints' / f'{VARIANT}_model' / f'best_mfft_{VARIANT}.pt'
if not ckpt_path.exists():
    print('No local checkpoint — downloading from HF...')
    ckpt_path = env.download_latest_checkpoint(
        PROJECT_ROOT / 'model' / 'checkpoints' / f'{VARIANT}_model',
        f'{VARIANT}_model'
    )

if ckpt_path is not None:
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    print(f'Loaded checkpoint: {ckpt_path.name}')
else:
    print('WARNING: No checkpoint found — model is randomly initialized')

model.eval()
print(f'Model: MFFT-{VARIANT}, params={sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Cell 3: Define cross-domain datasets
# These are the HELD-OUT datasets — never seen during training
CROSS_DOMAIN = {
    'dall-e3': ('ai_generated', 'DALL-E3'),
    'celebdf-v2image-dataset': ('deepfake', 'Celeb_V2'),
    'genimage-ai': ('ai_generated', 'genimage_ai'),
}

# Find mounts
input_root = Path('/kaggle/input')
slug_map = {}
for base in [input_root, input_root / 'datasets']:
    if not base.exists():
        continue
    for owner in base.iterdir():
        if not owner.is_dir():
            continue
        for child in owner.iterdir():
            if child.is_dir():
                slug_map[child.name] = child

print(f'Available mounts: {list(slug_map.keys())}')
for slug in CROSS_DOMAIN:
    status = 'FOUND' if slug in slug_map else 'MISSING'
    print(f'  {slug}: {status}')

In [ ]:
# Cell 4: Evaluate on each cross-domain dataset
from torchvision import transforms
from PIL import Image

transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

IMG_EXT = {'.jpg', '.jpeg', '.png', '.webp'}
LABEL_MAP = {'real': 0, 'ai_generated': 1, 'deepfake': 2}

def evaluate_dataset(mount_path, subdir, true_label, max_samples=2000):
    """Evaluate model on a dataset directory."""
    img_dir = mount_path / subdir
    if not img_dir.exists():
        img_dir = mount_path

    # Collect images
    images = []
    for dirpath, _, filenames in os.walk(str(img_dir)):
        for fname in filenames:
            if Path(fname).suffix.lower() in IMG_EXT:
                images.append(os.path.join(dirpath, fname))
                if len(images) >= max_samples:
                    break
        if len(images) >= max_samples:
            break

    if not images:
        return None

    # Evaluate
    correct = 0
    total = 0
    probs_all = []
    labels_all = []

    with torch.no_grad():
        for img_path in tqdm(images, desc=f'Evaluating'):
            try:
                img = Image.open(img_path).convert('RGB')
                tensor = transform(img).unsqueeze(0).to(device)
                logits = model(tensor)
                probs = torch.softmax(logits, dim=-1)
                pred = logits.argmax(dim=-1).item()
                true_int = LABEL_MAP[true_label]
                if pred == true_int:
                    correct += 1
                total += 1
                probs_all.append(probs[0].cpu().numpy())
                labels_all.append(true_int)
            except Exception as e:
                continue

    acc = correct / total * 100 if total > 0 else 0
    return {
        'accuracy': acc,
        'correct': correct,
        'total': total,
        'true_label': true_label,
    }

print('Evaluation function ready')

In [ ]:
# Cell 5: Run cross-domain evaluation
results = {}

for slug, (label, subdir) in CROSS_DOMAIN.items():
    print(f'\n{"="*60}')
    print(f'Evaluating: {slug} (label: {label})')
    print(f'{"="*60}')

    mount = slug_map.get(slug)
    if mount is None:
        # Try fuzzy match
        for s, p in slug_map.items():
            if slug.replace('-', '') in s.replace('-', '') or s.replace('-', '') in slug.replace('-', ''):
                mount = p
                break
    if mount is None:
        print(f'  SKIPPED: mount not found')
        continue

    result = evaluate_dataset(mount, subdir, label, max_samples=2000)
    if result:
        results[slug] = result
        print(f'  Accuracy: {result["accuracy"]:.2f}% ({result["correct"]}/{result["total"]})')
    else:
        print(f'  SKIPPED: no images found')

In [ ]:
# Cell 6: Summary table
if results:
    df = pd.DataFrame(results).T
    df['accuracy'] = df['accuracy'].round(2)
    print('\n=== CROSS-DOMAIN VALIDATION RESULTS ===')
    print(df[['accuracy', 'correct', 'total', 'true_label']].to_string())
    print(f'\nMean cross-domain accuracy: {df["accuracy"].mean():.2f}%')

    # Save results
    out_dir = PROJECT_ROOT / 'paper' / 'result' / ('verify' if not torch.cuda.is_available() else 'full_scale') / 'cross_domain'
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_dir / 'cross_domain_results.csv')
    print(f'\nSaved to {out_dir / "cross_domain_results.csv"}')

    # Upload to HF
    env.upload_to_hf(out_dir / 'cross_domain_results.csv', env.hf_results_repo, 'results/cross_domain/cross_domain_results.csv')
else:
    print('No results to show')